# Household Electricity Consumption Prediction Using Traditional Machine Learning
### Academic Capstone Project — Time-Series Regression with From-Scratch Algorithms
**Dataset**: UCI Individual Household Electric Power Consumption (2,075,259 observations)  
**Primary Target**: `Global_active_power` (kW)  
**Authors**: Team 114 (Iterate) / IIT Madras BS Degree


## 1. Project Objectives & Workflow Overview
1. **Large-scale Time Series Preprocessing**: Parse 2.07M timestamps, audit & impute missing observations (1.25%) using forward-fill.
2. **Temporal Exploratory Data Analysis**: Diurnal patterns, weekly dynamics, seasonal trends, correlation matrices, and ACF/PACF lag decay.
3. **Causal Feature Engineering**: Construct temporal, cyclical, autoregressive lag, and rolling statistics with **zero future leakage**.
4. **Chronological Train/Validation/Test Split**: Preserving temporal causality (70% Train, 15% Val, 15% Test).
5. **From-Scratch ML Regression Implementations**:
   - Linear Regression (Normal Equations via SVD Pseudoinverse)
   - Ridge Regression ($L_2$ Regularized Normal Equations)
   - Lasso Regression ($L_1$ Cyclic Coordinate Descent with Soft-Thresholding)
   - Decision Tree Regressor (Recursive MSE Variance Reduction)
   - Random Forest Regressor (Bagging + Feature Subspace Sampling)
   - Extra Trees Regressor (Extremely Randomized Trees with Uniform Cut-Points)
   - Benchmark against Gradient Boosting (GBDT).
6. **Feature Ablation Experiments**: Experiments A, B, C, D comparing electrical, temporal, lag, and rolling predictors.
7. **Capstone Investigation**: Horizon sensitivity ($E_1$: 10m, $E_2$: 30m, $E_3$: 1h, $E_4$: 6h, $E_5$: 24h).
8. **Error Analysis & Feature Importance**: Visualizing residual distributions, homoscedasticity, and split gains.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src modules are imported
sys.path.append(os.path.abspath("."))
from src.preprocessing import load_raw_dataset, audit_missing_values, handle_missing_values, chronological_split
from src.feature_engineering import build_experiment_features
from src.models_scratch import (
    LinearRegressionScratch,
    RidgeRegressionScratch,
    LassoRegressionScratch,
    DecisionTreeRegressorScratch,
    RandomForestRegressorScratch,
    ExtraTreesRegressorScratch,
    calculate_metrics
)
from src.experiments import get_model_suite

# Set clean aesthetic plot styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120
print("[*] All libraries and from-scratch models loaded successfully.")


## 2. Ingestion & Missing Value Handling
The dataset contains 2,075,259 one-minute electrical measurements. Approximately 1.25% of rows (25,979 timestamps) contain missing observations represented as `?`.
Because this is time-series data, **forward-fill (`ffill`)** is used to replicate the last known sensor reading, strictly preventing future data leakage.


In [ ]:
dataset_path = "dataset.txt"
df_raw = load_raw_dataset(dataset_path)

# Audit missingness
audit_df = audit_missing_values(df_raw)
display(audit_df)

# Handle missing observations using causal forward-fill
df_clean = handle_missing_values(df_raw, method="ffill")
print(f"Dataset ready: {len(df_clean):,} clean records.")
df_clean.head(5)


## 3. Exploratory Data Analysis (EDA) & The 9 Core Visualizations
We examine the consumption dynamics across multiple temporal scales:
- Diurnal cycle (24-hour hourly profile)
- Day of the month variations (days 1-31)
- Weekly dynamics (weekday vs. weekend)
- Seasonal dynamics (monthly consumption)
- Electrical inter-feature correlation
- Autoregressive decay (Autocorrelation Function across 60 minutes)
- Signal smoothing (Raw vs 1h vs 24h rolling averages)


In [ ]:
from src.eda import (
    plot_active_power_distribution,
    plot_power_over_time,
    plot_hourly_consumption_pattern,
    plot_daily_consumption_pattern,
    plot_weekly_consumption_pattern,
    plot_monthly_consumption_pattern,
    plot_correlation_heatmap,
    plot_lag_correlation,
    plot_rolling_mean_comparison
)

# Render and display all 9 EDA plots
plot_active_power_distribution(df_clean, "figures/fig01_active_power_distribution.png")
plot_hourly_consumption_pattern(df_clean, "figures/fig03_hourly_consumption_pattern.png")
plot_weekly_consumption_pattern(df_clean, "figures/fig05_weekly_consumption_pattern.png")
plot_monthly_consumption_pattern(df_clean, "figures/fig06_monthly_consumption_pattern.png")
plot_correlation_heatmap(df_clean, "figures/fig07_correlation_heatmap.png")
plot_lag_correlation(df_clean, max_lag=60, save_path="figures/fig08_lag_correlation_acf_pacf.png")


## 4. Causal Feature Engineering (Zero-Leakage)
We construct 44 engineered predictors categorized into:
1. **Base Electrical Variables**: Reactive Power, Voltage, Intensity, Sub-meterings 1-3, Remainder active energy.
2. **Temporal & Cyclical**: Hour, Minute, Day, Day of week, Month, Weekend flag, plus $\sin/\cos$ transformations.
3. **Autoregressive Lags**: Lags at $1	ext{m}, 10	ext{m}, 30	ext{m}, 60	ext{m}, 1440	ext{m}$ (24 hours).
4. **Rolling Statistics**: Rolling mean, standard deviation, minimum, and maximum over $10	ext{m}, 30	ext{m}, 60	ext{m}$ windows (computed on $y_{t-1}$ to prevent self-target leakage).


In [ ]:
X_full, y_full, feature_names = build_experiment_features(
    df_clean, experiment_type="D", target_col="Global_active_power", horizon_minutes=1
)
print(f"Total features created: {len(feature_names)}")
print("Engineered feature list:", feature_names)


## 5. From-Scratch Algorithm Benchmark
We benchmark our 6 from-scratch algorithms against Gradient Boosting on the chronological test partition.
Evaluation Metrics:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **MAPE** (Mean Absolute Percentage Error)
- **$R^2$** (Coefficient of Determination)


In [ ]:
# Display precomputed benchmark results
comp_df = pd.read_csv("results/model_comparison_summary.csv")
display(comp_df.sort_values("RMSE"))


## 6. Capstone Investigation: Forecasting Horizons ($E_1 	o E_5$)
We analyze how predictive accuracy evolves as the forecasting horizon increases:
- **$E_1$**: Next 10 minutes ($t+10$)
- **$E_2$**: Next 30 minutes ($t+30$)
- **$E_3$**: Next 1 hour ($t+60$)
- **$E_4$**: Next 6 hours ($t+360$)
- **$E_5$**: Next 24 hours ($t+1440$)


In [ ]:
horizon_df = pd.read_csv("results/forecast_horizon_results.csv")
display(horizon_df)


## 7. Error Analysis & Interpretability (Figures 10–14)


In [ ]:
from IPython.display import Image, display

# Display Key Plots
display(Image("figures/fig10_actual_vs_predicted.png"))
display(Image("figures/fig12_forecast_horizon_vs_rmse.png"))
display(Image("figures/fig13_feature_importance.png"))
display(Image("figures/fig14_model_comparison.png"))


## 8. Key Findings & Conclusions
1. **Lasso & GBDT Leadership**: At 1-minute resolution, Lasso Regression (from scratch) achieved an impressive RMSE of 0.1835 kW and MAPE of 9.26%, closely matching Gradient Boosting (0.1707 kW RMSE).
2. **Horizon Degradation**: Linear models degrade quickly beyond 30 minutes, whereas Random Forest and GBDT retain robust predictive structure.
3. **24-Hour Periodicity Recovery**: Prediction error peaks around 6 hours but exhibits measurable recovery at 24 hours due to the strong diurnal rhythm of domestic electricity usage.
